
# Business Complaint Intelligence Platform - DistilBERT

### FYP: Complaint Category Classification

This Google Colab notebook trains a **5-class DistilBERT text classifier** for the complaint categories defined in the FYP proposal:

- `billing`
- `delivery`
- `product_quality`
- `technical`
- `customer_service`

The notebook covers dataset loading, label mapping, cleaning, train/validation/test splitting, DistilBERT fine-tuning, evaluation, confusion matrix, and live complaint prediction.

> **Recommended:** enable a GPU in **Runtime → Change runtime type → T4 GPU** before training.



## 1. Install dependencies

The versions below are selected for a Colab workflow using Hugging Face Transformers and Datasets.


In [ ]:
!pip -q install -U transformers datasets accelerate evaluate scikit-learn pandas matplotlib seaborn

## 2. Imports and configuration

In [ ]:

import os
import json
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset, Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

MODEL_NAME = "distilbert/distilbert-base-uncased"
OUTPUT_DIR = "/content/complaint-distilbert"
MAX_LENGTH = 256

LABELS = [
    "billing",
    "delivery",
    "product_quality",
    "technical",
    "customer_service",
]

LABEL2ID = {label: i for i, label in enumerate(LABELS)}
ID2LABEL = {i: label for label, i in LABEL2ID.items()}

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))



## 3. Optional: upload your own verified complaints

Your CSV should contain exactly these columns:

```text
text,label
"I was charged twice",billing
"My parcel has not arrived",delivery
...
```

The label must be one of the five final classes above. Your own Pakistani SMB complaint data should eventually become the main training/evaluation source for the FYP.


In [ ]:

# Run this cell only when you want to upload your own CSV.
# from google.colab import files
# uploaded = files.upload()
# CUSTOM_FILE = next(iter(uploaded))
# custom_df = pd.read_csv(CUSTOM_FILE)
# display(custom_df.head())


## 4. Load public complaint/support datasets

In [ ]:

def load_ecommerce():
    ds = load_dataset("Ataur77/ecommerce-customer-support", split="train")
    df = ds.to_pandas()[["Complaint", "Category"]].rename(
        columns={"Complaint": "text", "Category": "source_category"}
    )
    mapping = {
        "Payments": "billing",
        "Refunds": "billing",
        "Order Tracking": "delivery",
        "Product Issues": "product_quality",
        "Technical Issues": "technical",
    }
    df["label"] = df["source_category"].map(mapping)
    return df[df["label"].notna()][["text", "label"]]


def load_bitext():
    ds = load_dataset(
        "bitext/Bitext-customer-support-llm-chatbot-training-dataset",
        split="train",
    )
    df = ds.to_pandas()[["instruction", "category"]].rename(
        columns={"instruction": "text", "category": "source_category"}
    )
    mapping = {
        "INVOICE": "billing",
        "PAYMENT": "billing",
        "REFUND": "billing",
        "CANCELLATION_FEE": "billing",
        "DELIVERY": "delivery",
        "SHIPPING_ADDRESS": "delivery",
        "FEEDBACK": "customer_service",
    }
    df["label"] = df["source_category"].map(mapping)
    return df[df["label"].notna()][["text", "label"]]


ecommerce_df = load_ecommerce()
bitext_df = load_bitext()

print("E-commerce mapped rows:", len(ecommerce_df))
print("Bitext mapped rows:", len(bitext_df))

public_df = pd.concat([ecommerce_df, bitext_df], ignore_index=True)
display(public_df.head())


## 5. Add the starter custom complaint examples (optional)

In [ ]:

# These are the 25 starter examples shipped with the project.
# Replace/add them with your own verified dataset for the final FYP.

starter_csv = """text,label
I was charged twice for the same order,billing
The amount on my invoice is higher than expected,billing
I requested a refund but the money has not arrived,billing
My card was charged for an order that failed,billing
The bill contains a charge I do not recognize,billing
My package has not arrived and the tracking has stopped,delivery
The courier delivered my order to the wrong address,delivery
I paid for express delivery but the parcel is late,delivery
The tracking number shows delivered but I received nothing,delivery
The courier lost my shipment,delivery
The product arrived damaged and cannot be used,product_quality
The item has a manufacturing defect,product_quality
The headphones I received have only one working side,product_quality
The food container was leaking when it arrived,product_quality
The product does not match the description,product_quality
The checkout page keeps crashing,technical
The website shows an error whenever I submit the form,technical
The app freezes when I try to pay,technical
I cannot log in because the system keeps rejecting my password,technical
The tracking page is not loading,technical
Nobody replied to my complaint for three days,customer_service
The support agent closed my ticket without solving the problem,customer_service
I have contacted support several times with no response,customer_service
The customer service team gave me conflicting information,customer_service
I want to speak to a supervisor about this complaint,customer_service
"""

from io import StringIO
starter_df = pd.read_csv(StringIO(starter_csv))

# Use this only as a small demonstration set. Do not rely on it as the final benchmark.
public_plus_starter = pd.concat([public_df, starter_df], ignore_index=True)
print("Rows including starter examples:", len(public_plus_starter))


## 6. Clean and inspect the dataset

In [ ]:

def clean(df):
    df = df.copy()
    df["text"] = df["text"].astype(str).str.strip().str.replace(r"\s+", " ", regex=True)
    df["label"] = df["label"].astype(str).str.strip().str.lower()
    df = df[df["text"].str.len() >= 5]
    df = df[df["label"].isin(LABELS)]
    df = df.drop_duplicates(subset=["text", "label"])
    return df.reset_index(drop=True)

all_df = clean(public_plus_starter)

print("Total clean samples:", len(all_df))
print("\nClass distribution:")
display(all_df["label"].value_counts().reindex(LABELS, fill_value=0).to_frame("count"))

plt.figure(figsize=(8, 4))
sns.countplot(data=all_df, x="label", order=LABELS)
plt.xticks(rotation=25)
plt.title("Complaint Category Distribution")
plt.tight_layout()
plt.show()


## 7. Stratified train / validation / test split

In [ ]:

train_df, temp_df = train_test_split(
    all_df,
    test_size=0.20,
    random_state=SEED,
    stratify=all_df["label"],
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label"],
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print("\nTrain distribution:")
display(train_df["label"].value_counts().reindex(LABELS, fill_value=0).to_frame("count"))


## 8. Convert pandas data to Hugging Face datasets

In [ ]:

def to_hf(df):
    out = df.copy()
    out["labels"] = out["label"].map(LABEL2ID)
    return Dataset.from_pandas(out[["text", "labels"]], preserve_index=False)

dataset = DatasetDict({
    "train": to_hf(train_df),
    "validation": to_hf(val_df),
    "test": to_hf(test_df),
})

dataset


## 9. Tokenization

In [ ]:

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

tokenized = dataset.map(tokenize, batched=True)
collator = DataCollatorWithPadding(tokenizer=tokenizer)

tokenized


## 10. Load DistilBERT for 5-class classification

In [ ]:

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

print("Model loaded:", MODEL_NAME)


## 11. Evaluation metrics

In [ ]:

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="macro",
        zero_division=0,
    )
    weighted_f1 = precision_recall_fscore_support(
        labels,
        preds,
        average="weighted",
        zero_division=0,
    )[2]
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1,
        "weighted_f1": weighted_f1,
    }


## 12. Train DistilBERT

In [ ]:

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=25,
    save_total_limit=2,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

trainer.train()


## 13. Evaluate on the held-out test set

In [ ]:

test_output = trainer.predict(tokenized["test"])
y_true = test_output.label_ids
y_pred = np.argmax(test_output.predictions, axis=1)

print("Test metrics:")
print(json.dumps(compute_metrics((test_output.predictions, y_true)), indent=2))

print("\nClassification report:")
print(classification_report(
    y_true,
    y_pred,
    labels=list(range(len(LABELS))),
    target_names=LABELS,
    digits=4,
    zero_division=0,
))


## 14. Confusion matrix

In [ ]:

cm = confusion_matrix(y_true, y_pred, labels=list(range(len(LABELS))))

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=LABELS,
    yticklabels=LABELS,
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Complaint Category Confusion Matrix")
plt.xticks(rotation=25)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


## 15. Save the trained model

In [ ]:

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Saved model to:", OUTPUT_DIR)
print("Files:")
for root, dirs, files in os.walk(OUTPUT_DIR):
    for f in files:
        print(os.path.join(root, f))


## 16. Live complaint prediction

In [ ]:

def predict_complaint(text, top_k=3):
    trainer.model.eval()
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
    )
    inputs = {k: v.to(trainer.model.device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = trainer.model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)[0].cpu().numpy()

    order = np.argsort(probs)[::-1][:top_k]
    return pd.DataFrame({
        "category": [ID2LABEL[int(i)] for i in order],
        "confidence": [float(probs[i]) for i in order],
    })

complaint = "I paid for express delivery but my parcel is still not here."
print("Complaint:", complaint)
display(predict_complaint(complaint))


In [ ]:

# Try your own complaint here.
my_complaint = "The app keeps showing an error when I try to make a payment."
display(predict_complaint(my_complaint))



## 17. Download the trained model from Colab

This creates a ZIP file that can be downloaded and later used by your Django/Flask/Node integration layer.


In [ ]:

import shutil

MODEL_ZIP = "/content/complaint-distilbert.zip"
shutil.make_archive("/content/complaint-distilbert", "zip", OUTPUT_DIR)
print("Created:", MODEL_ZIP)

# In Google Colab, uncomment these lines to download to your computer.
# from google.colab import files
# files.download(MODEL_ZIP)



## 18. Recommended FYP experiment

For the thesis, run the notebook in two stages:

**Experiment A:** public datasets only. Report the resulting test metrics clearly as benchmark/domain-transfer performance.

**Experiment B:** add a larger, manually verified dataset of real Pakistani SMB complaints, then split that dataset into independent train/validation/test sets and report the final model metrics.

Do not mix your final test set into training. The test set should remain unseen until the end of training.

### Final five labels

| ID | Category |
|---:|---|
| 0 | billing |
| 1 | delivery |
| 2 | product_quality |
| 3 | technical |
| 4 | customer_service |
